In [1]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("cnn-digit-classifier")

2026/07/29 17:51:03 INFO mlflow.tracking.fluent: Experiment with name 'cnn-digit-classifier' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/swar/CNN/mlruns/1', creation_time=1785347463431, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785347463431, lifecycle_stage='active', name='cnn-digit-classifier', tags={}, trace_location=None, workspace='default'>

In [3]:
import mlflow
import mlflow.tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("cnn-digit-classifier")

# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

def build_model():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# Enable autologging (captures params, metrics, model automatically)
mlflow.tensorflow.autolog()

with mlflow.start_run(run_name="cnn-mnist-run1") as run:
    model = build_model()
    model.fit(x_train, y_train, epochs=5, batch_size=64,
              validation_split=0.1)

    test_loss, test_acc = model.evaluate(x_test, y_test)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_loss", test_loss)

    # Log model explicitly (autolog also does this, but being explicit
    # lets you control the signature/input example)
    mlflow.tensorflow.log_model(
        model,
        artifact_path="cnn-model",
        registered_model_name="digit-classifier-cnn"
    )

    print("Run ID:", run.info.run_id)

Epoch 1/5
842/844 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9358 - loss: 0.2082

844/844 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.9359 - loss: 0.2079 - val_accuracy: 0.9850 - val_loss: 0.0524
Epoch 2/5
843/844 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9790 - loss: 0.0679

844/844 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.9790 - loss: 0.0679 - val_accuracy: 0.9882 - val_loss: 0.0371
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9850 - loss: 0.0485

844/844 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.9850 - loss: 0.0485 - val_accuracy: 0.9895 - val_loss: 0.0368
Epoch 4/5
842/844 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9881 - loss: 0.0379

844/844 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.9881 - loss: 0.0379 - val_accuracy: 0.9913 - val_loss: 0.0356
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.9901 - loss: 0.0314 - val_accuracy: 0.9890 - val_loss: 0.0396
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step


2026/07/29 17:54:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9892 - loss: 0.0324


2026/07/29 17:54:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 17:54:51 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
Successfully registered model 'digit-classifier-cnn'.
2026/07/29 17:54:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digit-classifier-cnn, version 1
Created version '1' of model 'digit-classifier-cnn'.


Run ID: fd1008f678f6463d8c09da04931bee0d
🏃 View run cnn-mnist-run1 at: http://localhost:5000/#/experiments/1/runs/fd1008f678f6463d8c09da04931bee0d
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [4]:
import requests
import numpy as np
import tensorflow as tf
import json

(_, _), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
sample = x_test[0].reshape(1, 28, 28, 1).astype("float32") / 255.0

payload = {"inputs": sample.tolist()}

response = requests.post(
    "http://localhost:1234/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
)

print(response.json())
# Output: list of 10 probabilities per image -> take argmax for predicted digit

{'predictions': [[8.590729516644302e-11, 7.184655714809196e-08, 2.3256359327206155e-07, 4.4730271042681125e-07, 2.9878261323390554e-12, 1.2554990380664321e-10, 5.938151581780585e-15, 0.9999991655349731, 7.120759537571075e-10, 4.972632439148583e-08]]}


In [5]:
probs = response.json()["predictions"][0]
predicted_digit = int(np.argmax(probs))
print("Predicted digit:", predicted_digit)
print("Actual digit:", y_test[0])

Predicted digit: 7
Actual digit: 7
